# RGΦ Stage B2 — ejecución exacta en Colab

Este notebook **no reconstruye** el checkpoint y **no ejecuta producción** por defecto.

Orden vinculante:

1. montar Drive;
2. fijar el commit del repositorio;
3. verificar la entrada `t=185`;
4. ejecutar auditoría estructural completa;
5. ejecutar smoke tests;
6. piloto exacto `185 → 186`;
7. sólo después, reanudar hasta 211.

Una interrupción material no cambia el canon.


In [ ]:
# Configuración canónica
REPO_URL = "https://github.com/FacundoFirmenich/rgphi-radix-mobius-dynamics.git"
REPO_COMMIT = "98dcb9650ad559594b71a9c2957859a07f49ace6"
DRIVE_BASE = r"/content/drive/MyDrive/RGPHI_TAU5_217_CUSTODIA_2026-09-29/RGPHI_STAGE_B2_COLAB_2026-11-11"
CHECKPOINT_ROOT = DRIVE_BASE + "/checkpoint_t185"
CHECKPOINT_CUR = CHECKPOINT_ROOT + "/cur"
WORK_ROOT = DRIVE_BASE + "/work"
RECEIPTS_ROOT = DRIVE_BASE + "/receipts"

RUN_MODE = "pilot"      # "pilot" o "full"
EXECUTE_STAGE_B2 = False
FULL_CHECKPOINT_AUDIT = True
JOBS = 2
MERGE_JOBS = 2
MINIMUM_FREE_GB = 100.0


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
for p in [DRIVE_BASE, CHECKPOINT_CUR, WORK_ROOT, RECEIPTS_ROOT]:
    Path(p).mkdir(parents=True, exist_ok=True)

print("Drive base:", DRIVE_BASE)
print("Checkpoint:", CHECKPOINT_CUR)
print("Work:", WORK_ROOT)


In [ ]:
# Recursos del runtime
import os, shutil, platform, json
local = shutil.disk_usage("/content")
drive_space = shutil.disk_usage("/content/drive")
runtime = {
    "cpu_count": os.cpu_count(),
    "platform": platform.platform(),
    "local_free_gib": local.free / 1024**3,
    "drive_free_gib": drive_space.free / 1024**3,
}
print(json.dumps(runtime, indent=2))
if runtime["drive_free_gib"] < MINIMUM_FREE_GB:
    raise RuntimeError("Drive no alcanza el mínimo canónico de espacio libre.")


In [ ]:
# Clonar y fijar byte-for-byte el commit canónico
import os, subprocess, pathlib, shutil
repo = pathlib.Path("/content/rgphi-radix-mobius-dynamics")
if repo.exists():
    shutil.rmtree(repo)
subprocess.run(["git", "clone", "--filter=blob:none", REPO_URL, str(repo)], check=True)
subprocess.run(["git", "-C", str(repo), "checkout", REPO_COMMIT], check=True)
head = subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip()
assert head == REPO_COMMIT, (head, REPO_COMMIT)
print("Pinned commit:", head)


In [ ]:
# Copiar identidad liviana del checkpoint a Drive
from pathlib import Path
import shutil, hashlib
repo = Path("/content/rgphi-radix-mobius-dynamics")
manifest = repo / "data/stage_b2/RGPHI_FRONTIER185_BUCKET_SHA256SUMS.txt"
identity = repo / "data/stage_b2/RGPHI_FRONTIER185_CHECKPOINT_MANIFEST.json"
shutil.copy2(manifest, Path(CHECKPOINT_ROOT) / manifest.name)
shutil.copy2(identity, Path(CHECKPOINT_ROOT) / identity.name)

def sha256(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for block in iter(lambda:f.read(1<<20),b""): h.update(block)
    return h.hexdigest()

assert sha256(manifest) == "dbd2f4adda52315e05261b9328c03f8a44d2a31b8e293f3dd3ff7b80b3d2a6b7"
print("Manifest identity PASS")


In [ ]:
# Inventario rápido: no calcula todavía los 128 hashes de 19,7 GB
import subprocess, sys
quick = Path("/content/quick_checkpoint_inventory.py")
quick.write_text('''#!/usr/bin/env python3
from pathlib import Path
import hashlib, json, os, re, sys

EXPECTED_BUCKETS = 128
EXPECTED_BYTES = 19_700_805_368
EXPECTED_STATES = 2_462_600_671
EXPECTED_MANIFEST_SHA = "dbd2f4adda52315e05261b9328c03f8a44d2a31b8e293f3dd3ff7b80b3d2a6b7"

def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

def main(root: str, manifest: str, output: str) -> int:
    root = Path(root)
    manifest = Path(manifest)
    rows = []
    for line in manifest.read_text().splitlines():
        if not line.strip():
            continue
        digest, name = line.split(maxsplit=1)
        name = name.strip().lstrip("*")
        m = re.search(r"b(\d{4})\.bin$", name)
        if not m:
            raise ValueError(name)
        rows.append((int(m.group(1)), digest, Path(name).name))
    rows.sort()

    errors = []
    if sha256(manifest) != EXPECTED_MANIFEST_SHA:
        errors.append("manifest_sha_mismatch")
    if len(rows) != EXPECTED_BUCKETS:
        errors.append(f"manifest_rows={len(rows)}")

    files = []
    total_bytes = 0
    for bucket, expected_sha, basename in rows:
        p = root / basename
        if not p.exists():
            errors.append(f"missing:{basename}")
            continue
        size = p.stat().st_size
        total_bytes += size
        files.append({"bucket": bucket, "name": basename, "bytes": size, "expected_sha256": expected_sha})
    if total_bytes != EXPECTED_BYTES:
        errors.append(f"total_bytes={total_bytes}")
    result = {
        "status": "INVENTORY_READY" if not errors else "BLOCKED",
        "root": str(root),
        "files_present": len(files),
        "total_bytes": total_bytes,
        "expected_states": EXPECTED_STATES,
        "errors": errors,
        "files": files,
    }
    Path(output).write_text(json.dumps(result, indent=2) + "\n")
    print(json.dumps({k: result[k] for k in ("status","files_present","total_bytes","errors")}, indent=2))
    return 0 if not errors else 3

if __name__ == "__main__":
    raise SystemExit(main(*sys.argv[1:4]))
''')
inventory = Path(RECEIPTS_ROOT) / "CHECKPOINT_INVENTORY.json"
cmd = [sys.executable, str(quick), CHECKPOINT_CUR, str(manifest), str(inventory)]
p = subprocess.run(cmd, text=True)
if p.returncode:
    raise RuntimeError(
        "Checkpoint t=185 ausente o incompleto. "
        "Subí b0000.bin...b0127.bin a checkpoint_t185/cur y repetí esta celda."
    )


In [ ]:
# Auditoría canónica completa: hashes + estructura de los 2.462.600.671 registros
# Esta celda lee aproximadamente 19,7 GB y debe ejecutarse antes de producción.
import subprocess, sys
validator = repo / "scripts/stage_b2/validate_checkpoint.py"
readiness = Path(RECEIPTS_ROOT) / "CHECKPOINT_READINESS_FULL.json"
mode = "full" if FULL_CHECKPOINT_AUDIT else "hash"
cmd = [
    sys.executable, str(validator), CHECKPOINT_ROOT,
    "--hash-manifest", str(manifest),
    "--mode", mode,
    "--output", str(readiness),
]
subprocess.run(cmd, check=True)
print(readiness.read_text())


In [ ]:
# Smoke y determinismo del motor exacto
import subprocess, sys
subprocess.run([sys.executable, str(repo/"scripts/stage_b2/smoke_test.py")], check=True)
subprocess.run([sys.executable, str(repo/"scripts/stage_b2/determinism_test.py")], check=True)
print("ENGINE_SMOKE_AND_DETERMINISM_PASS")


## Producción

El piloto procesa sólo `185 → 186`.  
La ejecución completa reanuda sobre la misma carpeta `work/` y omite niveles ya aceptados.

**No cambies `EXECUTE_STAGE_B2` a `True` antes de que las dos celdas anteriores pasen.**


In [ ]:
# Ejecución exacta y reanudable
if not EXECUTE_STAGE_B2:
    print("PRODUCTION_NOT_AUTHORIZED: cambiá EXECUTE_STAGE_B2=True sólo tras preflight y smoke PASS.")
else:
    import subprocess, sys, os, shutil
    end_time = 186 if RUN_MODE == "pilot" else 211
    cmd = [
        sys.executable,
        str(repo/"scripts/stage_b2/run_stage_b2.py"),
        "--checkpoint-root", CHECKPOINT_ROOT,
        "--hash-manifest", str(manifest),
        "--work-root", WORK_ROOT,
        "--start-time", "185",
        "--end-time", str(end_time),
        "--jobs", str(JOBS),
        "--merge-jobs", str(MERGE_JOBS),
        "--minimum-free-gb", str(MINIMUM_FREE_GB),
    ]
    if FULL_CHECKPOINT_AUDIT:
        cmd.append("--full-checkpoint-audit")
    print("RUNNING:", " ".join(cmd))
    subprocess.run(cmd, check=True)


In [ ]:
# Estado transaccional
from pathlib import Path
for name in [
    "CHECKPOINT_READINESS_FULL.json",
    "STAGE_B2_LEDGER.json",
    "RGPHI_STAGE_B2_FINAL_RESULT.json",
]:
    candidates = [Path(RECEIPTS_ROOT)/name, Path(WORK_ROOT)/name]
    for p in candidates:
        if p.exists():
            print("\n==", p, "==")
            print(p.read_text()[:20000])


## Interpretación vinculante

- `HIT`: nuevo upper bound exacto por debajo de 212.
- `NO_HIT_THROUGH_211`: permite concluir \(	au_5=212\).
- Piloto o interrupción: conserva el último nivel completo, sin cambiar el canon.
- Un checkpoint histórico de otra profundidad **no** puede reemplazar `t=185`.
